# drfn-fix-wide-s22: finish the 1024-slot arm (bpe_aux s22; gmm + bpe_aux s33)

In [ ]:
!pip install -q tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_extra.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
print('pack', pack_src)
def stream(cmd, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if line.startswith('{"step"') or line.startswith("{'step'"): continue
        print(line, end='')
    return p.wait()

In [ ]:
print('exit', stream(['bash', 'run_extra.sh'], env=dict(os.environ)))

In [ ]:
rows = []
for d in sorted(glob.glob('runs/*_s*')):
    lg, fe = os.path.join(d, 'train_log.jsonl'), os.path.join(d, 'final_eval.json')
    if not os.path.exists(lg): continue
    ev = [json.loads(l) for l in open(lg) if '"dev_loss"' in l]
    if not ev: continue
    b = min(ev, key=lambda e: e['dev_loss'])
    f = json.load(open(fe)) if os.path.exists(fe) else {}
    b5 = f.get('beam5_nr4') or {}
    pr = (f.get('probe_root') or {}).get('all_types') or {}
    gp = (f.get('probe_root') or {}).get('gate_probe') or {}
    rows.append((os.path.basename(d), len(ev), b['dev_loss'], b['epoch'],
                 b5.get('chrF_char'), b5.get('BLEU'),
                 (f.get('probe_root') or {}).get('unseen_slot_accuracy'),
                 gp.get('mean_gate_rooted'), gp.get('mean_gate_rootless')))
fmt = lambda v, s='%.2f': 'n/a' if v is None else s % v
print('%-28s %4s %9s %4s %10s %8s %11s %15s' % (
    'run', 'ep', 'best loss', '@ep', 'chrF(b5)', 'BLEU(b5)', 'root unseen', 'gate(rt/null)'))
for r in rows:
    print('%-28s %4d %9.4f %4d %10s %8s %11s %15s' % (
        r[0], r[1], r[2], r[3], fmt(r[4]), fmt(r[5]),
        ('%.1f%%' % (100 * r[6])) if r[6] is not None else 'n/a',
        (fmt(r[7], '%.2f') + '/' + fmt(r[8], '%.2f')) if r[7] is not None else 'n/a'))
open('/kaggle/working/summary.txt', 'w').write(repr(rows))